# 🛰️ Sentinel-2 Super Resolution với SuperResolutionV1

Notebook này nâng cấp ảnh vệ tinh Sentinel-2 từ **10m/20m/60m** lên **1m/px** sử dụng mô hình deep learning **SuperResolutionV1**.

## ⚙️ Yêu cầu
- **GPU**: Vào `Runtime` → `Change runtime type` → Chọn **T4 GPU**
- **Google Account**: Cần xác thực Earth Engine

## 📋 Quy trình
1. Cài đặt dependencies
2. Xác thực Google Earth Engine
3. Tìm ảnh Sentinel-2 cloud-free
4. Chạy SuperResolutionV1 super resolution
5. Xem & tải kết quả

---
## 1️⃣ Cài đặt Dependencies

In [ ]:
# Cài đặt các package cần thiết
!pip -q install earthengine-api rioxarray

# Cài đặt SuperResolutionV1 (Gamma Earth)
# Lưu ý: URL có thể thay đổi khi có phiên bản mới
!pip -q install https://storage.googleapis.com/0x7ff601307fa5/superresolutionv1-20260129.1-cp312-cp312-linux_x86_64.whl

print('✅ Cài đặt hoàn tất!')

: 

---
## 2️⃣ Kiểm tra GPU

In [ ]:
# Kiểm tra GPU khả dụng
!nvidia-smi

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

: 

---
## 3️⃣ Xác thực Google Earth Engine

In [ ]:
import ee

# ⚠️ THAY ĐỔI: Điền Project ID của bạn
PROJECT_ID = 'your-gcp-project-id'  # <-- THAY ĐỔI Ở ĐÂY

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

# Kiểm tra
print(f'✅ Earth Engine đã khởi tạo (project: {PROJECT_ID})')
print(f'   Test: {ee.Number(1).getInfo()}')

---
## 4️⃣ Cấu hình khu vực và thời gian

In [ ]:
# ============================================
# ⚠️ THAY ĐỔI CÁC THAM SỐ NÀY
# ============================================

# Tọa độ trung tâm (VD: TP.HCM)
LATITUDE = 10.762622
LONGITUDE = 106.660172

# Khoảng thời gian
START_DATE = '2024-01-01'
END_DATE = '2024-06-01'

# Ngưỡng mây tối đa (%)
CLOUD_THRESHOLD = 20

# Bán kính buffer (mét)
BUFFER_METERS = 5000

print(f'📍 Tọa độ: {LATITUDE}, {LONGITUDE}')
print(f'📅 Thời gian: {START_DATE} → {END_DATE}')
print(f'☁️ Cloud max: {CLOUD_THRESHOLD}%')
print(f'📐 Buffer: {BUFFER_METERS}m')

---
## 5️⃣ Tìm ảnh Sentinel-2 Cloud-Free

In [ ]:
from datetime import datetime

# Tạo AOI
point = ee.Geometry.Point([LONGITUDE, LATITUDE])
aoi = point.buffer(BUFFER_METERS).bounds()

# Tìm ảnh Sentinel-2 SR Harmonized
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLD))
    .sort('CLOUDY_PIXEL_PERCENTAGE')
    .limit(20)
)

count = collection.size().getInfo()
print(f'🔍 Tìm thấy {count} ảnh phù hợp:\n')

# Hiển thị danh sách
images = collection.toList(20)
for i in range(min(count, 10)):
    img = ee.Image(images.get(i))
    props = img.getInfo()['properties']
    date_ms = props.get('system:time_start', 0)
    date_str = datetime.fromtimestamp(date_ms/1000).strftime('%Y-%m-%d') if date_ms else 'N/A'
    cloud = props.get('CLOUDY_PIXEL_PERCENTAGE', 'N/A')
    tile = props.get('MGRS_TILE', 'N/A')
    img_id = img.getInfo()['id']
    emoji = '☀️' if isinstance(cloud, (int, float)) and cloud < 5 else '⛅'
    print(f'  {emoji} [{i+1}] {date_str} | Cloud: {cloud:.1f}% | Tile: {tile}')
    print(f'       ID: {img_id}')

---
## 6️⃣ Chạy SuperResolutionV1 Super Resolution 🚀

Bước này sử dụng SuperResolutionV1 để nâng cấp 10 bands (B2-B12) từ 10m/20m → **1m/px**

In [ ]:
import superresolutionv1
import os

# Tạo thư mục output
OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================
# ⚠️ CHỌN NGÀY TỪ KẾT QUẢ TÌM KIẾM Ở TRÊN
# ============================================
SELECTED_DATE = '2024-02-15'  # <-- THAY ĐỔI NGÀY PHÙ HỢP

print(f'🚀 Bắt đầu SuperResolutionV1 super resolution...')
print(f'   Tọa độ: {LATITUDE}, {LONGITUDE}')
print(f'   Ngày:   {SELECTED_DATE}')
print(f'   Output: {OUTPUT_DIR}')
print()

# Chạy SuperResolutionV1
result = superresolutionv1.process(
    lat=LATITUDE,
    lon=LONGITUDE,
    date=SELECTED_DATE,
    output_dir=OUTPUT_DIR,
)

print(f'\n✅ SuperResolutionV1 hoàn tất!')

---
## 7️⃣ Xem kết quả

In [ ]:
import glob
import rasterio
import numpy as np
import matplotlib.pyplot as plt

# Tìm files output
output_files = glob.glob(f'{OUTPUT_DIR}/*.tif')
print(f'📁 Files output ({len(output_files)}):')
for f in output_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'   - {os.path.basename(f)} ({size_mb:.1f} MB)')

# Visualize
if output_files:
    with rasterio.open(output_files[0]) as src:
        data = src.read()
        print(f'\n📊 Shape: {data.shape}')
        print(f'   CRS: {src.crs}')
        print(f'   Resolution: {src.res}')

        # RGB visualization
        if data.shape[0] >= 3:
            fig, axes = plt.subplots(1, 2, figsize=(16, 6))

            # True Color RGB (B4=Red, B3=Green, B2=Blue)
            rgb = np.stack([
                np.clip((data[2].astype(float) - np.percentile(data[2], 2)) /
                        (np.percentile(data[2], 98) - np.percentile(data[2], 2)), 0, 1),
                np.clip((data[1].astype(float) - np.percentile(data[1], 2)) /
                        (np.percentile(data[1], 98) - np.percentile(data[1], 2)), 0, 1),
                np.clip((data[0].astype(float) - np.percentile(data[0], 2)) /
                        (np.percentile(data[0], 98) - np.percentile(data[0], 2)), 0, 1),
            ], axis=-1)

            axes[0].imshow(rgb)
            axes[0].set_title(f'SuperResolutionV1 — RGB True Color (1m/px)\n{data.shape[1]}x{data.shape[2]} pixels')
            axes[0].axis('off')

            # NDVI nếu có NIR band
            if data.shape[0] >= 4:
                nir = data[3].astype(float)
                red = data[2].astype(float)
                ndvi = np.where((nir + red) != 0, (nir - red) / (nir + red), 0)
                im = axes[1].imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
                axes[1].set_title('NDVI (1m/px)')
                axes[1].axis('off')
                plt.colorbar(im, ax=axes[1], shrink=0.8)

            plt.tight_layout()
            plt.show()

---
## 8️⃣ Tải kết quả về

In [ ]:
from google.colab import files

# Tải từng file
for f in output_files:
    print(f'⬇️ Đang tải: {os.path.basename(f)}')
    files.download(f)

print('\n✅ Tải xong! Kiểm tra thư mục Downloads.')

---
## 📚 Tài liệu

- **SuperResolutionV1**: [Gamma Earth](https://medium.com/@yosef.akhtman)
- **Sentinel-2 Bands**: [ESA](https://sentinels.copernicus.eu/web/sentinel/user-guides/sentinel-2-msi/resolutions/spatial)
- **Earth Engine API**: [Google Developers](https://developers.google.com/earth-engine)
- **Video gốc**: [YouTube](https://www.youtube.com/watch?v=X077bA4aqco)